# 🔍 AeroGuard TSLM — Notebook 2: Windowing & PyTorch DataLoader Pre-Check

This notebook validates the preprocessed 30-cycle telemetry observation windows, verifies the strict **Zero Data Leakage** engine partitions, and tests the lazy JSONL byte-offset PyTorch dataset loader.

### Objectives:
1. Inspect `data/processed/windows.jsonl` and `dataset_manifest.json`.
2. Verify physical engine partition integrity (Train: 1–70, Val: 71–80, Test: 81–100).
3. Test `training.dataset_loader.CMAPSSCoTDataset` lazy loading.
4. Verify normalization statistics fitting and persistence (`preprocessing.json`).
5. Inspect aerospace Chain-of-Thought (CoT) prompts and diagnostic rationales.

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from training.dataset_loader import CMAPSSCoTDataset, NormalizationStats

print(f"✔ Project root: {PROJECT_ROOT}")

## 1. Inspect Dataset Manifest & Split Partitions

The dataset manifest (`data/processed/dataset_manifest.json`) records the exact hashes, window parameters, and engine split assignments.

In [ ]:
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_manifest.json"
assert MANIFEST_PATH.is_file(), "dataset_manifest.json not found. Run preprocessing first."

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

print("=== Dataset Manifest Summary ===")
print(f"• Schema Version:      {manifest.get('schema_version')}")
print(f"• Raw Data SHA-256:    {manifest.get('raw_sha256')}")
print(f"• Window Size:         {manifest.get('window_size')} cycles")
print(f"• Window Stride:       {manifest.get('stride')} cycles")
print(f"• Total Windows:       {manifest.get('record_count')}")
print(f"• Split Window Counts: {manifest.get('split_counts')}")

## 2. Verify Zero-Leakage Engine Splitting

In zero-leakage time-series prognostics, engine IDs in the training set must **NEVER** appear in validation or testing.

In [ ]:
WINDOWS_PATH = PROJECT_ROOT / "data" / "processed" / "windows.jsonl"
assert WINDOWS_PATH.is_file(), "windows.jsonl not found. Run preprocessing first."

# Scan records to verify engine IDs per split
split_engines = {"train": set(), "validation": set(), "test": set()}

with open(WINDOWS_PATH) as f:
    for line in f:
        item = json.loads(line)
        split_engines[item["split"]].add(item["unit_number"])

print("Engine IDs per split:")
print(f"• Train ({len(split_engines['train'])} engines):      Units {min(split_engines['train'])} to {max(split_engines['train'])}")
print(f"• Validation ({len(split_engines['validation'])} engines): Units {min(split_engines['validation'])} to {max(split_engines['validation'])}")
print(f"• Test ({len(split_engines['test'])} engines):       Units {min(split_engines['test'])} to {max(split_engines['test'])}")

# Check for any overlapping engines
assert len(split_engines["train"].intersection(split_engines["validation"])) == 0, "Leakage detected between Train and Val!"
assert len(split_engines["train"].intersection(split_engines["test"])) == 0, "Leakage detected between Train and Test!"
assert len(split_engines["validation"].intersection(split_engines["test"])) == 0, "Leakage detected between Val and Test!"
print("✔ ZERO LEAKAGE CONFIRMED: All engine partitions are mutually exclusive!")

## 3. Test Lazy PyTorch Dataset Loader

The `CMAPSSCoTDataset` uses lazy byte-offset indexing to avoid materializing all sensor arrays in memory simultaneously.

In [ ]:
# 1. Initialize training dataset (fits normalization parameters on training engines only)
train_dataset = CMAPSSCoTDataset(
    jsonl_path=str(WINDOWS_PATH),
    split="train"
)

print(f"✔ Initialized Train Dataset: {len(train_dataset)} windows")

# 2. Inspect a sample window
sample = train_dataset[0]
print(f"• Sample Record ID:   {sample['record_id']}")
print(f"• Engine Unit:        #{sample['unit_number']}")
print(f"• Flight Cycle:       {sample['cycle']}")
print(f"• Ground Truth RUL:   {sample['rul']} cycles")
print(f"• Sensor Tensor Shape:{sample['sensor_series'].shape} (Channels=14, Window=30)")
print(f"• Sensor Tensor Dtype:{sample['sensor_series'].dtype}")

## 4. Inspect Normalization Statistics

Normalization parameters must be fitted strictly on training windows and reused for validation/testing.

In [ ]:
norm_stats = train_dataset.normalization
print("=== Normalization Statistics (Fitted on Training Set) ===")
print(f"• Channel count: {len(norm_stats.channels)}")
print(f"• Training samples per channel: {norm_stats.sample_count_per_channel}")

stats_df = pd.DataFrame({
    "Channel": norm_stats.channels,
    "Mean": norm_stats.means,
    "Std": norm_stats.scales
})
stats_df.head(10)

## 5. Inspect Chain-of-Thought (CoT) Diagnostic Annotations

Each record contains structured natural language text tailored for causal language model training.

In [ ]:
print("--- PROMPT ---")
print(sample["prompt"])
print("\n--- GROUND TRUTH DIAGNOSTIC RATIONALE ---")
print(sample["target"])
print("\n--- TARGET RESPONSE ---")
print(sample["target"])

## 6. PyTorch DataLoader Batching Pre-Check

Verify that `DataLoader` creates properly shaped batches with tensors ready for the patch encoder.

In [ ]:
loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
batch = next(iter(loader))

print(f"✔ Batch sensor tensor shape: {batch['sensor_series'].shape} [Batch=16, Channels=14, Window=30]")
print(f"✔ Batch RUL targets shape:   {batch['rul'].shape} [Batch=16]")
print(f"✔ Batch RUL target sample:   {batch['rul'][:5].tolist()}")